# Dataset explorer

Set `dataset_dir` to an existing Parquet dataset directory. Loading returns a Polars LazyFrame; only the preview cell reads rows. Run once in this environment: `pip install -e ".[notebooks,vis]"`. The `vis` extra includes Kaleido for the SVG renderer and static image export.

In [4]:
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import plotly.io as pio

from pathlib import Path

from comm_research.infra.paths import PROJECT_ROOT
from comm_research.infra.lake.parquet import scan_parquet
from comm_research.infra.tools.vis import *

In [5]:
DATA_DIR = PROJECT_ROOT / "data/lake/weather/era5/t_2m"  

In [12]:
def load_data(data_dir):

    path = Path(DATA_DIR)
    return scan_parquet(path).collect()

def save(df: pl.DataFrame | pl.LazyFrame, filepath: str | Path) -> Path:
    clean_dir = (PROJECT_ROOT / "data" / "clean").resolve()
    relative_path = Path(filepath)

    if relative_path.is_absolute():
        raise ValueError("filepath must be relative to data/clean")

    path = (clean_dir / relative_path).with_suffix(".parquet").resolve()
    
    if not path.is_relative_to(clean_dir):
        raise ValueError("filepath must stay within data/clean")

    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(df, pl.LazyFrame):
        df.sink_parquet(path, compression="zstd")
    else:
        df.write_parquet(path, compression="zstd")
    return path


In [16]:
df = scan_parquet(DATA_DIR).collect()

In [37]:
df

timestamp,latitude,longitude,variable,value,unit,year,month,day
"datetime[μs, UTC]",f64,f64,str,f64,str,i64,i64,i64
2000-01-01 00:00:00 UTC,32.5,-121.0,"""2m_temperature""",286.295898,"""K""",2000,1,1
2000-01-01 00:00:00 UTC,32.5,-120.75,"""2m_temperature""",286.323242,"""K""",2000,1,1
2000-01-01 00:00:00 UTC,32.5,-120.5,"""2m_temperature""",286.30957,"""K""",2000,1,1
2000-01-01 00:00:00 UTC,32.5,-120.25,"""2m_temperature""",286.24707,"""K""",2000,1,1
2000-01-01 00:00:00 UTC,32.5,-120.0,"""2m_temperature""",286.143555,"""K""",2000,1,1
…,…,…,…,…,…,…,…,…
2026-09-30 23:00:00 UTC,35.0,-115.0,"""2m_temperature""",303.90271,"""K""",2026,9,30
2026-09-30 23:00:00 UTC,35.0,-114.75,"""2m_temperature""",305.982788,"""K""",2026,9,30
2026-09-30 23:00:00 UTC,35.0,-114.5,"""2m_temperature""",305.05896,"""K""",2026,9,30


### SoCal Subseasonal Temperature

In [27]:
coordinates = (
    pd.DataFrame(df[['latitude', 'longitude']].unique())
    .rename(columns={0:'lat', 1:'lon'})
)

In [46]:
for row in coordinates.iterrows():
    id = row[0]
    lat = row[1].get('lat')
    lon = row[1].get('lon')

    sub_df = df.filter(
        (pl.col("latitude") == lat) & 
        (pl.col("longitude") == lon)
    )

    weekly = (
        sub_df.sort("timestamp")
        .group_by_dynamic("timestamp", every="1w")
        .agg(
            pl.col("latitude").first().alias("lat"),
            pl.col("longitude").first().alias("lon"),
            pl.col("value").mean().alias("2m_mean"),
            pl.col("value").max().alias("2m_max"),
            pl.col("value").min().alias("2m_min")
        )
    )

    save_dir = PROJECT_ROOT / "data" / "clean" / "weather" / "era5" / "t_2m" / "socal"
    save_dir.mkdir(parents=True, exist_ok=True)

    weekly.write_csv(save_dir / f"socal_temp_2m_{id}.csv")
    print(f'socal_temp_2m_{id}.csv written')

ColumnNotFoundError: unable to find column "lat"; valid columns: ["timestamp", "latitude", "longitude", "variable", "value", "unit", "year", "month", "day"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
DF ["timestamp", "latitude", "longitude", "variable", ...]; PROJECT */9 COLUMNS